# 공통(Common) 메트릭 데모 — SDK & API

이 노트북은 모든 에이전트 유형에 공통으로 적용되는 **공통 메트릭**을 두 가지 방식으로 계산한다.

- **SDK**: 메트릭 클래스를 직접 임포트해 `EvalContext`에 대해 계산한다.
- **API**: FastAPI 메트릭 서버의 엔드포인트에 HTTP로 요청해 계산한다.

대상 메트릭: `llm_judge`, `bertscore`, `p95_latency`, `token_usage`

## 사전 준비

```bash
uv sync --extra server --extra t2s
```

이 프로젝트의 가상환경 커널로 이 노트북을 실행한다.

> 참고: 심판(judge)은 기본적으로 오프라인 스텁(`lexical_overlap_judge`)을 쓰므로 LLM API 키가 없어도 실행된다. SDK와 API가 같은 스텁 심판을 쓰기 때문에 두 방식의 점수가 일치한다(단, `bertscore` 제외).

## 0. 데이터셋

하드코딩된 간단한 질의·응답 데이터셋이다. 각 항목은 질문(`input`), 에이전트 응답(`output`), 정답(`expected`), 그리고 성능 메타데이터(`latency_ms`, `tokens`)를 갖는다.

In [ ]:
DATASET = [
    {"input": "프랑스의 수도는 어디인가요?", "output": "프랑스의 수도는 파리입니다.", "expected": "파리", "latency_ms": 820.0, "tokens": 45},
    {"input": "물의 화학식은 무엇인가요?", "output": "물의 화학식은 H2O입니다.", "expected": "H2O", "latency_ms": 1250.0, "tokens": 60},
    {"input": "지구에서 가장 큰 대양은?", "output": "가장 큰 대양은 태평양입니다.", "expected": "태평양", "latency_ms": 2100.0, "tokens": 52},
]
for i, row in enumerate(DATASET):
    print(i, row)

---
# Part 1. SDK

메트릭 클래스를 직접 임포트해 계산한다. 먼저 임포트와 `EvalContext` 구성, 헬퍼를 준비한다.

In [ ]:
from agent_eval.core.contracts import EvalContext, MetaKey
from agent_eval.judges.backend import FunctionJudge, lexical_overlap_judge
from agent_eval.metrics.common import BertScore, LLMJudgeMetric, P95Latency, TokenUsage

# 데이터셋 전체를 러너로 집계해 대표값과 95% 신뢰구간(CI)을 구하는 헬퍼.
# API 응답의 "aggregate"와 정확히 같은 계산이다(러너 하나가 두 곳에서 재사용된다).
from agent_eval.core.gate import GatePolicy
from agent_eval.core.suite import Suite
from agent_eval.offline.runner import evaluate


def aggregate(metric, ctxs):
    suite = Suite("demo", metric.name, [metric], GatePolicy())
    return evaluate(suite, ctxs).aggregates[0]


def run_sdk(metric, ctxs):
    """각 항목을 개별 채점한 뒤 데이터셋 집계를 출력한다."""
    print(f"[SDK] {metric.name}")
    for i, ctx in enumerate(ctxs):
        r = metric.score(ctx)
        print(f"  #{i}: score={r.score:.3f}  passed={r.passed}  error={r.error}")
    agg = aggregate(metric, ctxs)
    print(f"  ▶ 집계 value={agg.value:.3f}  95% CI=[{agg.ci_low:.3f}, {agg.ci_high:.3f}]  n={agg.n}")

# 공통 메트릭은 최종 응답(output)과 메타데이터를 읽는다.
contexts = [
    EvalContext(
        input=row["input"],
        output=row["output"],
        expected=row["expected"],
        metadata={MetaKey.LATENCY_MS: row["latency_ms"], MetaKey.TOKENS: row["tokens"]},
    )
    for row in DATASET
]

# 오프라인 스텁 심판(실제 사용 시엔 LLMJudge로 교체).
judge = FunctionJudge(lexical_overlap_judge)
print("준비 완료:", len(contexts), "개 컨텍스트")

## 1-1. `llm_judge` — 응답 품질 심판

질문(`input`)과 응답(`output`)을, 있다면 정답(`expected`)과 함께 심판에게 넘겨 주관적 품질 점수(0~1)를 받는다.

In [ ]:
run_sdk(LLMJudgeMetric(judge), contexts)

## 1-2. `bertscore` — 정답과의 의미적 유사도

응답(`output`)과 정답(`expected`)의 임베딩 유사도(BERTScore F1). 실제로는 `bert-score` 패키지(torch)가 필요하므로, 여기서는 동작 방식을 보이기 위해 간단한 스텁 scorer를 주입한다.

In [ ]:
# 데모용 스텁 scorer: 문자 집합 자카드 유사도. 실제 사용 시엔 BertScore(lang="ko")처럼 scorer 없이 생성한다.
def stub_scorer(cands, refs):
    out = []
    for c, r in zip(cands, refs):
        cs, rs = set(c), set(r)
        out.append(len(cs & rs) / len(cs | rs) if (cs | rs) else 1.0)
    return out

run_sdk(BertScore(scorer=stub_scorer), contexts)

## 1-3. `p95_latency` — 꼬리(p95) 지연시간

각 항목은 `metadata['latency_ms']`를 그대로 돌려주고, 집계에서 95백분위수(p95)를 구한다. **낮을수록 좋다.**

In [ ]:
run_sdk(P95Latency(), contexts)

## 1-4. `token_usage` — 평균 토큰 사용량

`metadata['tokens']`의 평균. **낮을수록 좋다.**

In [ ]:
run_sdk(TokenUsage(), contexts)

---
# Part 2. API

동일한 데이터셋을 이번에는 HTTP 엔드포인트로 계산한다. 먼저 서버를 띄운다.

In [ ]:
# API 파트: 백그라운드 스레드에서 실제 FastAPI 서버를 띄우고, httpx로 진짜 HTTP 요청을 보낸다.
# (환경변수를 설정하지 않으면 서버도 SDK와 동일한 오프라인 스텁 심판을 쓴다 → 점수가 일치한다.)
import threading
import time

import httpx
import uvicorn

from agent_eval.server.app import create_app

PORT = 8077
BASE_URL = f"http://127.0.0.1:{PORT}"

if "server" not in globals():
    server = uvicorn.Server(uvicorn.Config(create_app(), host="127.0.0.1", port=PORT, log_level="warning"))
    threading.Thread(target=server.run, daemon=True).start()
    while not server.started:
        time.sleep(0.1)
print("API 서버 준비 완료:", BASE_URL)
print("health:", httpx.get(f"{BASE_URL}/health").json())

def call_api(path, contexts, params=None):
    """엔드포인트에 contexts를 POST하고, 개별 결과와 집계를 출력한다."""
    payload = {"contexts": contexts}
    if params:
        payload["params"] = params
    resp = httpx.post(BASE_URL + path, json=payload)
    print(f"[API] POST {path} → {resp.status_code}")
    data = resp.json()
    if resp.status_code != 200:
        print("  오류:", data.get("detail"))
        return data
    for i, item in enumerate(data["results"]):
        print(f"  #{i}: score={item['score']:.3f}  passed={item['passed']}  error={item['error']}")
    agg = data["aggregate"]
    print(f"  ▶ 집계 value={agg['value']:.3f}  95% CI=[{agg['ci_low']:.3f}, {agg['ci_high']:.3f}]  n={agg['n']}")
    return data

## 2-1. `POST /common/llm_judge`

각 컨텍스트는 `input`, `output`(선택적으로 `expected`)를 담는다. SDK의 1-1과 점수가 일치해야 한다.

In [ ]:
ctx = [{"input": r["input"], "output": r["output"], "expected": r["expected"]} for r in DATASET]
call_api("/common/llm_judge", ctx)

## 2-2. `POST /common/bertscore`

서버에 `bert-score`(torch)가 설치되어 있지 않으면 **501**을 돌려주며, 설치 방법 힌트를 함께 준다(오류 모델 시연). 설치되어 있다면 실제 BERTScore를 계산한다.

In [ ]:
ctx = [{"output": r["output"], "expected": r["expected"]} for r in DATASET]
call_api("/common/bertscore", ctx)

## 2-3. `p95_latency` · `token_usage` — API 엔드포인트 없음

이 두 메트릭은 입력/출력을 채점하지 않고 하니스가 채워준 메타데이터만 읽기 때문에 **서버 엔드포인트가 없다**. 위 SDK 1-3 / 1-4처럼 라이브러리로 계산한다.